# 49. Pydantic Data Validation — `pydantic`
**Python Libraries Lab · Phase 3 · Utilities & Testing**

_Turn messy input into clean, typed, validated Python objects._

✅ **Runs in Colab** as-is.

Data from forms, files and APIs is messy. A price might arrive as the text four ninety nine, an email might be missing, a quantity might be negative. Pydantic lets you describe the shape you expect with a class and type hints. It converts what it safely can, rejects what it cannot, and tells you exactly which field is wrong and why.

**In this notebook:** You will validate ShopEasy orders arriving as JSON. You will see automatic conversion, readable error messages, field limits, a custom validator for Indian pin codes, nested models, and exporting clean data back to JSON.

**How to use:** run each cell in order with **Shift + Enter**, or use **Runtime → Run all**. Read the short note above each cell first, then change the code and run it again: that is how you learn.

## Setup
The next cell makes sure the library is installed (Colab already has most of them; it takes a few seconds).

In [ ]:
!pip install -q pydantic
import importlib.metadata as md
for pkg in ['pydantic']:
    print(pkg, md.version(pkg))

## Step 1 · A first model
Type hints define the shape. Safe conversions happen automatically.

In [ ]:
from pydantic import BaseModel, Field, ValidationError, field_validator, EmailStr
from datetime import date

class Product(BaseModel):
    name: str
    price: float
    in_stock: bool = True

p = Product(name="Steel Bottle", price="499", in_stock="yes")
print(p)
print(type(p.price), p.price)

## Step 2 · Clear errors for bad data
All problems are reported at once, with the field and the reason.

In [ ]:
try:
    Product(name=None, price="four hundred")
except ValidationError as e:
    for err in e.errors():
        print(f"{err['loc'][0]:<6} -> {err['msg']}")

## Step 3 · Rules with Field and validators
Limits with `Field`. A custom `field_validator` checks the 6-digit pin code.

In [ ]:
class Address(BaseModel):
    line1: str = Field(min_length=3)
    city: str
    pincode: str

    @field_validator("pincode")
    @classmethod
    def six_digits(cls, v):
        v = v.replace(" ", "")
        if not (v.isdigit() and len(v) == 6):
            raise ValueError("pincode must be 6 digits")
        return v

class Item(BaseModel):
    sku: str = Field(pattern=r"^[A-Z]-\d{2}$")
    qty: int = Field(gt=0, le=10)
    price: float = Field(gt=0)

print(Address(line1="12 MG Road", city="Pune", pincode="411 001"))
try:
    Item(sku="b01", qty=0, price=499)
except ValidationError as e:
    print([f"{x['loc'][0]}: {x['msg']}" for x in e.errors()])

## Step 4 · Nested models from JSON
A whole order, validated in one call, with computed values.

In [ ]:
class Order(BaseModel):
    order_id: int
    email: EmailStr
    placed: date
    items: list[Item] = Field(min_length=1)
    address: Address
    coupon: str | None = None

    @property
    def total(self):
        return sum(i.qty * i.price for i in self.items)

raw = """{"order_id": 1001, "email": "asha@example.com", "placed": "2026-11-19",
          "items": [{"sku": "B-01", "qty": 2, "price": 499}, {"sku": "N-04", "qty": "3", "price": 149}],
          "address": {"line1": "12 MG Road", "city": "Pune", "pincode": "411001"}}"""
order = Order.model_validate_json(raw)
print(order.placed.strftime("%d %b %Y"), "|", len(order.items), "items | total Rs.", order.total)

## Step 5 · Export clean data
`model_dump` gives a dictionary, `model_dump_json` gives JSON text. `model_json_schema` describes the model.

In [ ]:
print(order.model_dump(exclude={"address"}))
print(order.model_dump_json(include={"order_id", "email"}))
print(list(Order.model_json_schema()["properties"]))

## ✍️ Your turn
Try to create `Item(sku='B-01', qty=12, price=100)`. Catch the `ValidationError` and store the **name of the failing field** (a string) in `answer`.

Write your code in the next cell, run it, then run the check cell.

In [ ]:
answer = None  # your code here

In [ ]:
def check():
    return answer == "qty", f"Expected 'qty', got {answer!r}"

# ▶ Run this cell to check your answer
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

### Stuck?
The next cell holds a solution. In Colab its code is hidden: double-click the cell to see it, or just run it.

In [ ]:
# @title 👀 Solution (double-click to show the code) { display-mode: "form" }
try:
    Item(sku="B-01", qty=12, price=100)
except ValidationError as e:
    answer = e.errors()[0]["loc"][0]

# Check again with the solution
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

## Recap
- **BaseModel**
- **Type hints**
- **ValidationError**
- **Field constraints**
- **field_validator**
- **model_dump**

**Next:** 50. Command-line Apps (`click · typer`): [open in Colab](https://colab.research.google.com/github/aashaysocial27-maker/python-libraries-colab/blob/main/notebooks/50_click_typer.ipynb)

## 💻 Run it on your own computer
1. Install Python 3.10+ from python.org (Windows: tick **Add python.exe to PATH**).
2. Open a terminal (Windows: *Command Prompt*; Mac: *Terminal*) and run:
```bash
mkdir python-lab && cd python-lab
python -m venv .venv
# Windows:  .venv\Scripts\activate      Mac/Linux:  source .venv/bin/activate
pip install pydantic
pip install notebook
jupyter notebook
```
3. In Colab choose **File → Download → Download .ipynb**, then open that file in the Jupyter page that opened in your browser.

Lines that start with `!` or `%%` are notebook commands. In a plain terminal, type `!` commands without the `!`.